# 1. Price Chart + Buy Signals (1 trade per day enforced)


## Install libraries

In [3]:
!pip install ta


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
!pip install loguru


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [5]:
!pip install backtrader


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [6]:
!pip install datetime


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
pip install TA-Lib


[notice] A new release of pip is available: 25.0 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Import libraries

In [3]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange

## Import Bitcoin Futures Contract Historical data 

In [4]:
data_15m = pd.read_csv("futures_klines_data/BTCUSDT_15m_2025.csv")
data_1h = pd.read_csv("futures_klines_data/BTCUSDT_1h_2025.csv")

## Technical Strategy 

In [8]:
import pandas as pd
import numpy as np
from ta.trend import EMAIndicator, MACD
from ta.momentum import RSIIndicator
from ta.volatility import AverageTrueRange

class Strategy:
    def __init__(self, data_15m, data_1h):
        """
        Enhanced strategy class with complete indicator calculation
        data_15m: DataFrame with 15-minute OHLCV data (must contain close_time)
        data_1h: DataFrame with 1-hour OHLCV data (must contain close_time)
        """
        # Clean and prepare data
        self.data_15m = self._prepare_data(data_15m.copy(), '15m')
        self.data_1h = self._prepare_data(data_1h.copy(), '1h')
        
    def _prepare_data(self, df, timeframe):
        """Prepare and validate data"""
        # Check essential columns
        essential_cols = ['open', 'high', 'low', 'close', 'volume', 'close_time']
        missing_cols = [col for col in essential_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing columns in {timeframe} data: {missing_cols}")
        
        # Convert to datetime if needed
        if not pd.api.types.is_datetime64_any_dtype(df['close_time']):
            df['close_time'] = pd.to_datetime(df['close_time'])
        
        # Clean data
        df_clean = df.dropna(subset=essential_cols, how='any')
        df_clean = df_clean[df_clean['volume'] > 0]
        df_clean = df_clean.sort_values('close_time')
        
        # Validate
        if df_clean.empty:
            raise ValueError(f"{timeframe} data is empty after cleaning")
        if len(df_clean) < 50:
            print(f"Warning: {timeframe} data has only {len(df_clean)} points")
            
        return df_clean.reset_index(drop=True)
    
    def calculate_15m_indicators(self):
        """Calculate 15m timeframe indicators"""
        df = self.data_15m
        
        # EMAs
        df['EMA9_15m'] = EMAIndicator(df['close'], 9).ema_indicator()
        df['EMA21_15m'] = EMAIndicator(df['close'], 21).ema_indicator()
        
        # MACD (requires minimum 26 periods)
        if len(df) >= 26:
            macd = MACD(df['close'], window_slow=26, window_fast=12, window_sign=9)
            df['MACD_15m'] = macd.macd()
            df['MACD_Signal_15m'] = macd.macd_signal()
            df['MACD_Hist_15m'] = macd.macd_diff()
        else:
            df[['MACD_15m', 'MACD_Signal_15m', 'MACD_Hist_15m']] = np.nan
        
        # RSI (requires minimum 14 periods)
        df['RSI_14_15m'] = RSIIndicator(df['close'], 14).rsi() if len(df) >= 14 else np.nan
        
        # ATR (requires minimum 14 periods)
        df['ATR_15m'] = AverageTrueRange(
            high=df['high'],
            low=df['low'],
            close=df['close'],
            window=14
        ).average_true_range() if len(df) >= 14 else np.nan
        
        return df.dropna().copy()
        
    def calculate_1h_indicators(self):
        """Calculate 1h timeframe indicators"""
        df = self.data_1h
        
        # EMAs
        df['EMA20_1h'] = EMAIndicator(df['close'], 20).ema_indicator()
        df['EMA50_1h'] = EMAIndicator(df['close'], 50).ema_indicator()
        df['EMA200_1h'] = EMAIndicator(df['close'], 200).ema_indicator() if len(df) >= 200 else np.nan
        
        # MACD
        if len(df) >= 26:
            macd = MACD(df['close'], window_slow=26, window_fast=12, window_sign=9)
            df['MACD_1h'] = macd.macd()
            df['MACD_Signal_1h'] = macd.macd_signal()
            df['MACD_Hist_1h'] = macd.macd_diff()
        else:
            df[['MACD_1h', 'MACD_Signal_1h', 'MACD_Hist_1h']] = np.nan
        
        # RSI
        df['RSI_14_1h'] = RSIIndicator(df['close'], 14).rsi() if len(df) >= 14 else np.nan
        df['RSI_7_1h'] = RSIIndicator(df['close'], 7).rsi() if len(df) >= 7 else np.nan
        
        # ATR
        df['ATR_1h'] = AverageTrueRange(
            high=df['high'],
            low=df['low'],
            close=df['close'],
            window=14
        ).average_true_range() if len(df) >= 14 else np.nan
        
        return df.dropna().copy()
        
    def calculate_all_indicators(self):
        """Calculate indicators for both timeframes"""
        df_15m = self.calculate_15m_indicators()
        df_1h = self.calculate_1h_indicators()
        
        if df_15m.empty or df_1h.empty:
            raise ValueError("Insufficient data after indicator calculation")
            
        return df_15m, df_1h

# Usage Example:
if __name__ == "__main__":
    # Assuming you have data_15m and data_1h DataFrames
    strategy = Strategy(data_15m, data_1h)
    data_15m_indicators, data_1h_indicators = strategy.calculate_all_indicators()
    
    print("15m Data with Indicators:")
    print(data_15m_indicators[['close_time', 'close', 'EMA9_15m', 'EMA21_15m', 'RSI_14_15m']].tail())
    
    print("\n1h Data with Indicators:")
    print(data_1h_indicators[['close_time', 'close', 'EMA20_1h', 'EMA50_1h', 'RSI_14_1h']].tail())

15m Data with Indicators:
                   close_time    close      EMA9_15m     EMA21_15m  RSI_14_15m
81995 2025-05-03 18:59:59.999  96103.6  96117.933195  96151.077061   46.405961
81996 2025-05-03 19:14:59.999  96159.3  96126.206556  96151.824600   49.889801
81997 2025-05-03 19:29:59.999  96219.8  96144.925245  96158.004182   53.430796
81998 2025-05-03 19:44:59.999  96244.2  96164.780196  96165.840166   54.817516
81999 2025-05-03 19:59:59.999  96312.0  96194.224157  96179.127423   58.514210

1h Data with Indicators:
                   close_time     close       EMA20_1h       EMA50_1h  \
20995 2025-05-24 11:59:59.999  109152.2  108526.184975  108933.303283   
20996 2025-05-24 12:59:59.999  108741.3  108546.672120  108925.773743   
20997 2025-05-24 13:59:59.999  108564.3  108548.350966  108911.598302   
20998 2025-05-24 14:59:59.999  108913.0  108583.079445  108911.653270   
20999 2025-05-24 15:59:59.999  108889.2  108612.233784  108910.772750   

       RSI_14_1h  
20995  54.711645

## Risk Management

In [9]:
class RiskManagement:
    def __init__(self, tp_percent=0.02, sl_percent=0.01, risk_per_trade=0.01):
        """
        tp_percent: Take profit percentage (2% default)
        sl_percent: Stop loss percentage (1% default)
        risk_per_trade: % of capital to risk per trade (1% default)
        """
        self.tp_percent = tp_percent
        self.sl_percent = sl_percent
        self.risk_per_trade = risk_per_trade
        self.current_balance = 10000  # Starting balance
        
    def calculate_tp_sl_prices(self, entry_price, position_type):
        """Calculate take profit and stop loss prices"""
        if position_type == 'BUY':
            tp_price = entry_price * (1 + self.tp_percent)
            sl_price = entry_price * (1 - self.sl_percent)
        else:  # SELL
            tp_price = entry_price * (1 - self.tp_percent)
            sl_price = entry_price * (1 + self.sl_percent)
        return tp_price, sl_price
        
    def calculate_position_size(self, entry_price, stop_loss_price):
        """Calculate position size based on risk parameters"""
        risk_amount = self.current_balance * self.risk_per_trade
        risk_per_coin = abs(entry_price - stop_loss_price)
        position_size = risk_amount / risk_per_coin
        return position_size
    
    def update_balance(self, entry_price, exit_price, position_size, position_type):
        """Update account balance after trade"""
        if position_type == 'BUY':
            pnl = (exit_price - entry_price) * position_size
        else:  # SELL
            pnl = (entry_price - exit_price) * position_size
        self.current_balance += pnl
        return pnl

# Testing the class
rs = RiskManagement(tp_percent=0.005, sl_percent=0.0025, risk_per_trade=0.0025)
entry_price = 101691

# Calculate TP/SL prices for SELL position
tp_price, sl_price = rs.calculate_tp_sl_prices(entry_price, "SELL")
print(f"SL Price: {sl_price:.2f} (Sell Short Stop Loss)")
print(f"TP Price: {tp_price:.2f} (Sell Short Take Profit)")

# Calculate position size (returns single value)
position_size = rs.calculate_position_size(entry_price, sl_price)
print(f"\nPosition Size: {position_size:.4f} coins")
print(f"Risk Amount: ${rs.current_balance * rs.risk_per_trade:.2f}")
print(f"Risk Per Coin: ${abs(entry_price - sl_price):.2f}")

# Example trade execution
exit_price = tp_price  # Assuming we hit TP
pnl = rs.update_balance(entry_price, exit_price, position_size, "SELL")
print(f"\nTrade P&L: ${pnl:.2f}")
print(f"New Balance: ${rs.current_balance:.2f}")

SL Price: 101945.23 (Sell Short Stop Loss)
TP Price: 101182.54 (Sell Short Take Profit)

Position Size: 0.0983 coins
Risk Amount: $25.00
Risk Per Coin: $254.23

Trade P&L: $50.00
New Balance: $10050.00


## SignalStrategy

In [12]:
class SignalStrategy:
    def __init__(self, data_15m, data_1h, risk_params={}):
        self.data_15m = data_15m.copy()
        self.data_1h = data_1h.copy()
        self.current_position = None
        self.trade_history = []
        self.risk_mgmt = RiskManagement(**risk_params)
        
        # Prepare time references with error handling
        try:
            self.data_15m['close_time'] = pd.to_datetime(self.data_15m['close_time'])
            self.data_1h['close_time'] = pd.to_datetime(self.data_1h['close_time'])
            self.data_15m['time_1h'] = self.data_15m['close_time'].dt.floor('h')
            self.data_1h['time_1h'] = self.data_1h['close_time'].dt.floor('h')
        except Exception as e:
            raise ValueError(f"Error preparing time references: {str(e)}")

    def _get_1h_context(self, close_time):
        """Safe method to get 1h context with proper error handling"""
        target_1h_time = close_time.floor('h')
        matching_1h = self.data_1h[self.data_1h['time_1h'] == target_1h_time]
        
        if len(matching_1h) == 0:
            # Return a Series of NaN values with the same structure as 1h data
            return pd.Series(index=self.data_1h.columns, dtype='float64')
        return matching_1h.iloc[0]

    def _check_entry_conditions(self, row_15m):
        """Safe entry condition checking"""
        try:
            row_1h = self._get_1h_context(row_15m['close_time'])
            
            # Check if we got valid 1h data (not all NaNs)
            if row_1h.isna().all():
                return None
                
            # Long conditions with safety checks
            long_cond = (
                pd.notna(row_15m['EMA9_15m']) and 
                pd.notna(row_15m['EMA21_15m']) and
                (row_15m['EMA9_15m'] > row_15m['EMA21_15m']) and
                pd.notna(row_15m['MACD_Hist_15m']) and
                (row_15m['MACD_Hist_15m'] > 0) and
                pd.notna(row_15m['RSI_14_15m']) and
                (row_15m['RSI_14_15m'] > 50) and
                pd.notna(row_1h['EMA20_1h']) and
                pd.notna(row_1h['EMA50_1h']) and
                (row_1h['EMA20_1h'] > row_1h['EMA50_1h']) and
                pd.notna(row_1h['MACD_Hist_1h']) and
                (row_1h['MACD_Hist_1h'] > 0)
            )
            
            # Short conditions with safety checks
            short_cond = (
                pd.notna(row_15m['EMA9_15m']) and 
                pd.notna(row_15m['EMA21_15m']) and
                (row_15m['EMA9_15m'] < row_15m['EMA21_15m']) and
                pd.notna(row_15m['MACD_Hist_15m']) and
                (row_15m['MACD_Hist_15m'] < 0) and
                pd.notna(row_15m['RSI_14_15m']) and
                (row_15m['RSI_14_15m'] < 50) and
                pd.notna(row_1h['EMA20_1h']) and
                pd.notna(row_1h['EMA50_1h']) and
                (row_1h['EMA20_1h'] < row_1h['EMA50_1h']) and
                pd.notna(row_1h['MACD_Hist_1h']) and
                (row_1h['MACD_Hist_1h'] < 0)
            )
            
            return 'BUY' if long_cond else 'SELL' if short_cond else None
            
        except Exception as e:
            print(f"Error checking entry conditions: {str(e)}")
            return None

    def run_simulation(self):
        """Robust simulation with error handling"""
        try:
            for i, row in self.data_15m.iterrows():
                try:
                    if not self.current_position:
                        signal = self._check_entry_conditions(row)
                        if signal:
                            entry_price = row['close']
                            tp, sl = self.risk_mgmt.calculate_tp_sl_prices(entry_price, signal)
                            pos_size = self.risk_mgmt.calculate_position_size(entry_price, sl)
                            
                            self.current_position = {
                                'type': signal,
                                'entry_time': row['close_time'],
                                'entry_price': entry_price,
                                'position_size': pos_size,
                                'tp_price': tp,
                                'sl_price': sl,
                                'entry_index': i
                            }
                    else:
                        exit_reason = self._check_exit_conditions(row)
                        if exit_reason:
                            exit_price = (
                                self.current_position['tp_price'] if exit_reason == 'TP' else
                                self.current_position['sl_price'] if exit_reason == 'SL' else
                                row['close']
                            )
                            
                            pnl = self.risk_mgmt.update_balance(
                                self.current_position['entry_price'],
                                exit_price,
                                self.current_position['position_size'],
                                self.current_position['type']
                            )
                            
                            trade_record = {
                                **self.current_position,
                                'exit_time': row['close_time'],
                                'exit_price': exit_price,
                                'exit_reason': exit_reason,
                                'pnl': pnl,
                                'pnl_pct': (pnl / (self.current_position['entry_price'] * 
                                          self.current_position['position_size']) * 100),
                                'current_balance': self.risk_mgmt.current_balance
                            }
                            
                            self.trade_history.append(trade_record)
                            self.current_position = None
                            
                except Exception as trade_error:
                    print(f"Error processing trade at index {i}: {str(trade_error)}")
                    continue
                    
            return pd.DataFrame(self.trade_history)
            
        except Exception as e:
            print(f"Simulation failed: {str(e)}")
            return pd.DataFrame()

# Example usage:
risk_params = {
    'tp_percent': 0.03,  # 3% take profit
    'sl_percent': 0.015,  # 1.5% stop loss
    'risk_per_trade': 0.01  # Risk 1% of capital per trade
}
strategy = SignalStrategy(data_15m_indicators, data_1h_indicators, risk_params)
results = strategy.run_simulation()

In [13]:
# Run with optimized parameters for 0.5% moves
params_05pct = {
    'tp_percent': 0.005,  # 0.5% target
    'sl_percent': 0.003,  # 0.3% stop loss
    'risk_per_trade': 0.01  # 1% risk per trade
}

strategy = SignalStrategy(data_15m_indicators, data_1h_indicators, params_05pct)
results = strategy.run_simulation()

# Analyze results
print(f"Total Trades: {len(results)}")
print(f"Win Rate: {(results['pnl'] > 0).mean():.1%}")
print(f"Average PnL: {results['pnl'].mean():.2f}")

Error processing candle at index 796: 'float' object has no attribute 'between'
Error processing candle at index 797: 'float' object has no attribute 'between'
Error processing candle at index 798: 'float' object has no attribute 'between'
Error processing candle at index 799: 'float' object has no attribute 'between'
Error processing candle at index 800: 'float' object has no attribute 'between'
Error processing candle at index 801: 'float' object has no attribute 'between'
Error processing candle at index 802: 'float' object has no attribute 'between'
Error processing candle at index 803: 'float' object has no attribute 'between'
Error processing candle at index 804: 'float' object has no attribute 'between'
Error processing candle at index 805: 'float' object has no attribute 'between'
Error processing candle at index 806: 'float' object has no attribute 'between'
Error processing candle at index 807: 'float' object has no attribute 'between'
Error processing candle at index 808: 'f

KeyboardInterrupt: 